In [1]:
from pathlib import Path
import pandas as pd

# Works when Jupyter starts from the project root or notebook folder.
csv_path = Path("evaluation_data/flan_evaluation.csv")
if not csv_path.exists():
    csv_path = Path("notebook/evaluation_data/flan_evaluation.csv")

df = pd.read_csv(csv_path, dtype=str).fillna("")

required = ["id", "title", "full_body", "summary", "reference_summary"]
assert set(required).issubset(df.columns), "Missing required columns"
assert df["id"].is_unique, "Duplicate article IDs"

for column in required:
    assert df[column].str.strip().ne("").all(), f"Empty values in {column}"

print(f"Ready to evaluate {len(df)} summary pairs.")
display(df[["id", "title", "summary", "reference_summary"]].head())

Ready to evaluate 17 summary pairs.


,id,title,summary,reference_summary
0,1,Kraken’s parent Payward is betting billions on...,"Payward is unifying trading, banking, asset ma...",Kraken parent Payward is investing billions in...
1,2,Binance deal gives Circle a boost in stablecoi...,The five-year deal could strengthen USDC’s rea...,Binance’s $100 million investment and expanded...
2,3,Bitget hacker moves $83 million in stolen XRP ...,Exchanges can restrict accounts that receive t...,The attacker in Bitget’s roughly $388 million ...
3,4,Bitcoin could soon get Zcash-style 'shielded' ...,"Shielded Bitcoin, a system modeled on Zcash th...",Researchers proposed Shielded Bitcoin to enabl...
4,5,Solana’s 150-millisecond settlement upgrade re...,Solana’s planned upgrade to make payments fina...,Solana’s Alpenglow upgrade is active on devnet...


In [2]:
%pip install rouge-score==0.1.2

Note: you may need to restart the kernel to use updated packages.


In [3]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

rows = []

for _, article in df.iterrows():
    scores = scorer.score(
        target=article["reference_summary"],
        prediction=article["summary"],
    )

    rows.append({
        "id": article["id"],
        "title": article["title"],
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rouge2_f1": scores["rouge2"].fmeasure,
        "rougeL_f1": scores["rougeL"].fmeasure,
    })

rouge_df = pd.DataFrame(rows)

display(rouge_df.round(4))
print("Average ROUGE scores:")
display(
    rouge_df[["rouge1_f1", "rouge2_f1", "rougeL_f1"]]
    .mean()
    .rename("mean_f1")
    .round(4)
)

,id,title,rouge1_f1,rouge2_f1,rougeL_f1
0,1,Kraken’s parent Payward is betting billions on...,0.3226,0.1667,0.3226
1,2,Binance deal gives Circle a boost in stablecoi...,0.3000,0.1034,0.2333
2,3,Bitget hacker moves $83 million in stolen XRP ...,0.3896,0.1867,0.1558
3,4,Bitcoin could soon get Zcash-style 'shielded' ...,0.3492,0.1967,0.2540
4,5,Solana’s 150-millisecond settlement upgrade re...,0.3117,0.0267,0.2597
5,6,XRP Ledger’s Batch upgrade slips to Oct. 9 aft...,0.5067,0.2192,0.2667
6,7,Circle and Tether step in to freeze hacker wal...,0.6230,0.4068,0.5574
7,8,Tokenization is moving faster than Washington,0.0282,0.0000,0.0282
8,9,Tether confirms minimal EQIBank exposure follo...,0.3182,0.0952,0.2727
9,10,Strategy proposes daily dividends to bring STR...,0.0444,0.0000,0.0444


Average ROUGE scores:


rouge1_f1    0.2679
rouge2_f1    0.1281
rougeL_f1    0.2182
Name: mean_f1, dtype: float64

In [4]:
%pip install bert-score==0.3.13

Note: you may need to restart the kernel to use updated packages.


In [5]:
from bert_score import score

(P, R, F1), bertscore_hash = score(
    cands=df["summary"].tolist(),
    refs=df["reference_summary"].tolist(),
    lang="en",
    model_type="roberta-large",
    device="cpu",
    batch_size=4,
    idf=False,
    rescale_with_baseline=False,
    return_hash=True,
    verbose=True,
)

bertscore_df = df[["id", "title"]].copy()
bertscore_df["bertscore_precision"] = P.cpu().numpy()
bertscore_df["bertscore_recall"] = R.cpu().numpy()
bertscore_df["bertscore_f1"] = F1.cpu().numpy()

display(bertscore_df.round(4))

print("Average BERTScore:")
display(
    bertscore_df[
        ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
    ].mean().round(4)
)

print("Configuration:", bertscore_hash)

/Users/mumu/Thesis Folder/Thesis Test/Pragma/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10942.67it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Co

calculating scores...
computing bert embedding.


100%|██████████| 9/9 [00:03<00:00,  2.74it/s]


computing greedy matching.


100%|██████████| 5/5 [00:00<00:00, 563.51it/s]

done in 3.30 seconds, 5.15 sentences/sec


,id,title,bertscore_precision,bertscore_recall,bertscore_f1
0,1,Kraken’s parent Payward is betting billions on...,0.9360,0.8649,0.8991
1,2,Binance deal gives Circle a boost in stablecoi...,0.9048,0.8795,0.8920
2,3,Bitget hacker moves $83 million in stolen XRP ...,0.8926,0.8878,0.8902
3,4,Bitcoin could soon get Zcash-style 'shielded' ...,0.9027,0.8884,0.8955
4,5,Solana’s 150-millisecond settlement upgrade re...,0.8935,0.8578,0.8753
5,6,XRP Ledger’s Batch upgrade slips to Oct. 9 aft...,0.9052,0.8898,0.8974
6,7,Circle and Tether step in to freeze hacker wal...,0.9487,0.9243,0.9363
7,8,Tokenization is moving faster than Washington,0.7498,0.8299,0.7878
8,9,Tether confirms minimal EQIBank exposure follo...,0.9511,0.8981,0.9238
9,10,Strategy proposes daily dividends to bring STR...,0.7977,0.8079,0.8028


Average BERTScore:


bertscore_precision    0.8729
bertscore_recall       0.8671
bertscore_f1           0.8695
dtype: float32

Configuration: roberta-large_L17_no-idf_version=0.3.12(hug_trans=5.14.1)


In [6]:
output_dir = csv_path.parent / "evaluation_results"
output_dir.mkdir(parents=True, exist_ok=True)

# Per-article scores
rouge_df.to_csv(output_dir / "rouge_scores.csv", index=False)
bertscore_df.to_csv(output_dir / "bertscore_scores.csv", index=False)

# Overall averages
averages = pd.concat([
    rouge_df[["rouge1_f1", "rouge2_f1", "rougeL_f1"]].mean(),
    bertscore_df[
        ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
    ].mean(),
])

averages.rename_axis("metric").reset_index(name="mean_score").to_csv(
    output_dir / "evaluation_summary.csv",
    index=False,
)

# Preserve BERTScore configuration for reproducibility
(output_dir / "bertscore_config.txt").write_text(
    bertscore_hash + "\n",
    encoding="utf-8",
)

print(f"Results saved to: {output_dir.resolve()}")

Results saved to: /Users/mumu/Thesis Folder/Thesis Test/Pragma/notebook/evaluation_data/evaluation_results
